In [1]:
import numpy as np
import pandas as pd

In [2]:
catalyst_df = pd.read_csv("datasets/catalyst_ds.csv")
market_data_df = pd.DataFrame(pd.read_csv("datasets/market_ds.csv"))
final_df = pd.DataFrame(pd.read_csv("datasets/final_ds_NCAR20.csv"))

In [3]:
# calculate immediate past stock price trends
def trend(market_data_df, ticker, eventDate):
    df = (market_data_df[market_data_df['Ticker'] == ticker].sort_values('Date').reset_index(drop=True))
    match = df.index[pd.to_datetime(df['Date'], utc = True).dt.strftime('%Y-%m-%d') == pd.to_datetime(eventDate, utc = True).strftime('%Y-%m-%d')]

    if len(match)==0 or match[0] < 60:
        return {
            'Stock_Trend_30d': np.nan,
            'Stock_Trend_60d': np.nan,
            'NBI_Trend_30d': np.nan,
            'NBI_Trend_60d': np.nan,
            'isDelisted': True
        }

    idx = match[0]
     
    # extract stock values at specified timepoints
    s0, s30, s60 = df.loc[[idx, idx-30, idx-60], 'Close'].values
    n0, n30, n60 = df.loc[[idx, idx-30, idx-60], 'NBI'].values

    stock30 = (s0-s30) / s30 if pd.notna(s30) and s30!=0 and s30 !="" else np.nan
    stock60 = (s0-s60) / s60 if pd.notna(s60) and s60!=0 and s60 !="" else np.nan
    nbi30 = (n0-n30) / n30 if pd.notna(n30) and n30!=0 and n30 !="" else np.nan
    nbi60 = (n0-n60) / n60 if pd.notna(n60) and n60!=0 and n60 !="" else np.nan

    # calculate trends
    return {
        'Stock_Trend_30d': stock30,
        'Stock_Trend_60d': stock60,
        'NBI_Trend_30d': nbi30,
        'NBI_Trend_60d': nbi60,
        'isDelisted': pd.isna(stock30) or pd.isna(stock60)
    }

In [4]:
trends_res = [trend(market_data_df, ticker, date)
                          for ticker, date in zip(final_df['company_ticker'], 
                                                  final_df['date'])]
trends_df = pd.DataFrame([res if isinstance(res, dict) else {} for res in trends_res])

trends_col = ['Stock_Trend_30d', 'Stock_Trend_60d','NBI_Trend_30d','NBI_Trend_60d','isDelisted']

for c in trends_col:
    final_df[c] = trends_df[c].values

In [5]:
final_df.to_csv("datasets/final_ds.csv", index = False)

In [6]:
final_df.columns

Index(['catalyst', 'date', 'company_name', 'company_ticker', 'drug_name',
       'nct_number', 'label', 'press_link', 'stage', 'polarity', 'NCAR20',
       'Total Revenue', 'Operating Revenue', 'Cash And Cash Equivalents',
       'Total Debt', 'Capital Expenditure', 'drug_count', 'Stock_Trend_30d',
       'Stock_Trend_60d', 'NBI_Trend_30d', 'NBI_Trend_60d', 'isDelisted'],
      dtype='object')